# ADA2 AlphaFold 3 results: scores and comparison to 3LGD

**How to use:** put each AlphaFold Server download (the `.zip`, or the unzipped folder) into `structure_prediction/af3_outputs/`, then **Run All**. Re-run after each new job finishes.

Sections: (1) score table, (2) ipTM/pTM chart, (3) PAE maps, (4) pLDDT profiles, (5) comparison to the 3LGD crystal structure, (6) seed variability, (7) human vs pig, (8) export.

Reminder: these are *confidence* scores of a prediction. They do not measure binding, activity or the effect of a mutation.

In [ ]:
import json, re, shlex, zipfile, itertools
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 60); pd.set_option("display.max_rows", 200)

# ---- paths (the repo root is found by looking for data/structures, so run from anywhere inside the repo) ----
HERE = Path.cwd()
REPO = next((p for p in [HERE, *HERE.parents] if (p / "data" / "structures").exists()), None)
assert REPO is not None, "Run this notebook from inside the repository (data/structures not found)."
PROJECT = REPO / "structure_prediction"
RUNS_DIR = PROJECT / "af3_outputs"
OUT_DIR = PROJECT / "af3_results"
XTAL_CIF = REPO / "data" / "structures" / "human_ADA2_3LGD_assembly1.cif"
POSMAP_CSV = PROJECT / "human_pig_position_map_DRAFT.csv"
for d in (RUNS_DIR, OUT_DIR): d.mkdir(parents=True, exist_ok=True)

CONTACT_CUTOFF = 4.0      # heavy-atom distance (A) for interface residues
XTAL_OFFSET = 26          # 3LGD residue number = UniProt number - 26
# UniProt number = AF residue index + offset  (AF residues are numbered 1..N within each chain)
UP_OFFSET = {("human", "mature"): 29, ("pig", "mature"): 24, ("human", "full"): 0, ("pig", "full"): 0}
ZN_LIGANDS = {"human": [112, 114, 356, 441], "pig": [107, 109, 353, 438]}   # UniProt numbering; H,H,H,D
HUMAN_LIG = [112, 114, 356, 441]
print("repo:", REPO); print("runs dir:", RUNS_DIR)

In [ ]:
# unzip any downloaded zips (once)
for z in RUNS_DIR.glob("*.zip"):
    dest = RUNS_DIR / z.stem
    if not dest.exists():
        with zipfile.ZipFile(z) as f: f.extractall(dest)
        print("unzipped", z.name)

def parse_meta(name):
    n = name.lower()
    species = "human" if "hum" in n else ("pig" if "pig" in n else "unknown")
    construct = "full" if "full" in n else "mature"
    zinc = "noZn" if "nozn" in n else ("Zn" if "zn" in n else "noZn")
    m = re.search(r"_s(\d+)$", n)
    template = "noTemplate" if "notemplate" in n else "template"
    return species, construct, zinc, (int(m.group(1)) if m else None), template

def find_jobs():
    jobs = {}
    for sj in RUNS_DIR.rglob("*_summary_confidences_*.json"):
        m = re.match(r"(.+)_summary_confidences_(\d+)\.json$", sj.name)
        prefix, idx = m.group(1), int(m.group(2))
        d = sj.parent
        name = re.sub(r"^fold_", "", prefix)
        jobs.setdefault(name, {})[idx] = dict(summary=sj, cif=d / f"{prefix}_model_{idx}.cif", full=d / f"{prefix}_full_data_{idx}.json")
    return jobs

JOBS = find_jobs()
print(len(JOBS), "jobs found:"); [print("  ", j, "->", len(m), "models") for j, m in sorted(JOBS.items())]

## 1. Score table (every model of every job)

In [ ]:
def load_json(p):
    with open(p) as f: return json.load(f)

rows = []
for name, models in JOBS.items():
    species, construct, zinc, seed, template = parse_meta(name)
    for idx, fp in sorted(models.items()):
        s = load_json(fp["summary"])
        r = dict(job=name, species=species, construct=construct, zinc=zinc, template=template, seed=seed, model=idx,
                 ptm=s.get("ptm"), iptm=s.get("iptm"), ranking_score=s.get("ranking_score"),
                 has_clash=s.get("has_clash"), fraction_disordered=s.get("fraction_disordered"))
        cp = s.get("chain_pair_pae_min")
        r["chain_pair_pae_min"] = cp[0][1] if cp else None
        if fp["full"].exists():
            fd = load_json(fp["full"])
            pae = np.array(fd["pae"]); ch = np.array(fd["token_chain_ids"])
            a, b = ch == "A", ch == "B"
            r["mean_pae_within"] = float(np.mean([pae[np.ix_(a, a)].mean(), pae[np.ix_(b, b)].mean()]))
            r["mean_pae_between"] = float(np.mean([pae[np.ix_(a, b)].mean(), pae[np.ix_(b, a)].mean()]))
            pl = np.array(fd["atom_plddts"]); r["mean_plddt"] = float(pl.mean())
        rows.append(r)
MODELS = pd.DataFrame(rows).sort_values(["species", "template", "zinc", "construct", "seed", "model"]).reset_index(drop=True)
BEST = MODELS.loc[MODELS.groupby("job").ranking_score.idxmax()].reset_index(drop=True)   # top-ranked model per job
print("Top-ranked model per job:")
display(BEST.round(3))

**What to look for:** ipTM above ~0.8 = confident chain placement; pTM above ~0.8 = confident overall fold; `has_clash` should be 0; `mean_pae_between` low (few Å) = confident relative placement of the two chains. Compare human vs pig for the same zinc setting.

## 2. ipTM / pTM per job

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
for k, col in enumerate(["iptm", "ptm"]):
    labels = list(BEST.job)
    ax[k].bar(range(len(BEST)), BEST[col], color=["tab:blue" if s == "human" else "tab:orange" for s in BEST.species], alpha=.7)
    for i, j in enumerate(labels):
        v = MODELS[MODELS.job == j][col]
        ax[k].scatter([i] * len(v), v, color="k", s=10, zorder=3)
    ax[k].set_xticks(range(len(BEST))); ax[k].set_xticklabels(labels, rotation=75, ha="right", fontsize=7)
    ax[k].axhline(.8, ls="--", c="grey", lw=.8); ax[k].set_ylim(0, 1); ax[k].set_title(col + " (bar = top model, dots = all 5)")
plt.tight_layout(); plt.show()

## 3. PAE maps (top model per job)

In [ ]:
n = len(BEST); cols = min(3, n); rws = int(np.ceil(n / cols))
fig, axs = plt.subplots(rws, cols, figsize=(5 * cols, 4.4 * rws), squeeze=False)
for ax in axs.ravel(): ax.axis("off")
for ax, (_, r) in zip(axs.ravel(), BEST.iterrows()):
    fp = JOBS[r.job][int(r.model)]["full"]
    if not fp.exists(): continue
    fd = load_json(fp); pae = np.array(fd["pae"]); ch = np.array(fd["token_chain_ids"])
    im = ax.imshow(pae, cmap="Greens_r", vmin=0, vmax=30); ax.axis("on")
    for b in np.where(ch[1:] != ch[:-1])[0]: ax.axhline(b + .5, c="k", lw=.6); ax.axvline(b + .5, c="k", lw=.6)
    ax.set_title(f"{r.job}\niptm={r.iptm:.2f} model {int(r.model)}", fontsize=8)
fig.colorbar(im, ax=axs, label="expected position error (A)", shrink=.6); plt.show()

**What to look for:** dark = confident. Dark off-diagonal blocks (chain A vs chain B) = confident dimer placement. Bright stripes = flexible regions or loops; note their residue numbers.

## 4. Load structures, position map, 3LGD reference

In [ ]:
def _num(x, default=np.nan):
    try: return float(x)
    except: return default

def read_atoms(path):
    cols, rows = [], []
    with open(path) as f:
        for line in f:
            if line.startswith("_atom_site."): cols.append(line.strip().split(".", 1)[1]); continue
            if cols and (line.startswith("ATOM") or line.startswith("HETATM")):
                try: rows.append(shlex.split(line))
                except ValueError: rows.append(line.split())
            elif cols and rows and (line.startswith("#") or line.startswith("loop_")): break
    df = pd.DataFrame(rows, columns=cols)
    out = pd.DataFrame(dict(group=df["group_PDB"], chain=df["label_asym_id"], comp=df["label_comp_id"], atom=df["label_atom_id"],
                            x=df["Cartn_x"].astype(float), y=df["Cartn_y"].astype(float), z=df["Cartn_z"].astype(float),
                            b=df["B_iso_or_equiv"].astype(float)))
    seq = df["label_seq_id"].map(_num)
    if "auth_seq_id" in df: seq = seq.fillna(df["auth_seq_id"].map(_num))
    out["resid"] = seq
    if "label_alt_id" in df: out = out[df["label_alt_id"].isin([".", "?", "A"])]
    return out[out.atom.map(lambda a: not str(a).startswith("H"))].reset_index(drop=True)

def prot_chains(atoms): return sorted(atoms[atoms.group == "ATOM"].chain.unique())

def ca_table(atoms):
    ca = atoms[(atoms.group == "ATOM") & (atoms.atom == "CA")]
    return {c: ca[ca.chain == c].reset_index(drop=True) for c in prot_chains(atoms)}

pm = pd.read_csv(POSMAP_CSV).dropna(subset=["human_pos", "pig_pos"])
PIG2HUM = dict(zip(pm.pig_pos.astype(int), pm.human_pos.astype(int)))
HUM2PIG = {v: k for k, v in PIG2HUM.items()}
IDENT = {int(h): bool(i) for h, i in zip(pm.human_pos, pm.identical)}

def human_pos_of(species, up):
    return up if species == "human" else PIG2HUM.get(up)

XTAL_ATOMS = read_atoms(XTAL_CIF); XTAL_CA = ca_table(XTAL_ATOMS)
print("3LGD protein chains:", list(XTAL_CA), "| residues:", {c: len(v) for c, v in XTAL_CA.items()})
print("Zn atoms in 3LGD:", (XTAL_ATOMS.comp == "ZN").sum())

STRUCT = {}
for _, r in BEST.iterrows():
    fp = JOBS[r.job][int(r.model)]["cif"]
    if not fp.exists(): print("missing cif", fp); continue
    at = read_atoms(fp); STRUCT[r.job] = dict(atoms=at, ca=ca_table(at), species=r.species, construct=r.construct, zinc=r.zinc, seed=r.seed, template=r.template)
    print(r.job, "chains:", prot_chains(at), "| Zn atoms:", (at.comp == "ZN").sum())

## 5. pLDDT profiles (top model per job), on human numbering so human and pig line up

In [ ]:
def plddt_by_human_pos(s):
    out = {}
    for c, ca in s["ca"].items():
        off = UP_OFFSET[(s["species"], s["construct"])]
        for resid, b in zip(ca.resid, ca.b):
            hp = human_pos_of(s["species"], int(resid) + off)
            if hp is not None: out.setdefault(hp, []).append(b)
    return pd.Series({k: np.mean(v) for k, v in out.items()}).sort_index()

fig, ax = plt.subplots(figsize=(14, 4))
for j, s in STRUCT.items():
    p = plddt_by_human_pos(s)
    ax.plot(p.index, p.values, lw=.9, label=j, ls="-" if s["species"] == "human" else "--")
for lo, hi, lab, c in [(30, 100, "dimerization", "#ccd"), (127, 185, "PRB domain", "#dcc"), (204, 211, "substrate site", "#cdc")]:
    ax.axvspan(lo, hi, color=c, alpha=.5); ax.text(lo, 52, lab, fontsize=7)
for z in HUMAN_LIG: ax.axvline(z, c="r", lw=.5)
ax.axhline(90, c="grey", ls=":"); ax.axhline(70, c="grey", ls=":")
ax.set_ylim(40, 100); ax.set_xlabel("human ADA2 UniProt position (red = Zn ligands)"); ax.set_ylabel("pLDDT")
ax.legend(fontsize=6, ncol=2); plt.tight_layout(); plt.show()

**What to look for:** pLDDT above 90 very high, 70 to 90 confident, 50 to 70 low, below 50 very low (often disordered). Dips in the same place in human and pig usually reflect flexible loops, not species differences.

## 6. Comparison with the 3LGD crystal structure

In [ ]:
def kabsch(P, Q):
    Pc, Qc = P.mean(0), Q.mean(0)
    U, S, Vt = np.linalg.svd((P - Pc).T @ (Q - Qc))
    d = np.sign(np.linalg.det(Vt.T @ U.T)); R = Vt.T @ np.diag([1, 1, d]) @ U.T
    return R, Qc - R @ Pc
def apply(R, t, X): return X @ R.T + t
def rmsd(A, B): return float(np.sqrt(((A - B) ** 2).sum(1).mean()))

def matched_pairs(s, chain_map):
    # returns per-chain arrays of (human_pos, model_xyz, xtal_xyz)
    off = UP_OFFSET[(s["species"], s["construct"])]; out = {}
    for mc, xc in chain_map.items():
        ca = s["ca"][mc]; xca = XTAL_CA[xc].set_index("resid")
        recs = []
        for resid, x, y, z in zip(ca.resid, ca.x, ca.y, ca.z):
            hp = human_pos_of(s["species"], int(resid) + off)
            if hp is None: continue
            xid = hp - XTAL_OFFSET
            if xid in xca.index:
                xr = xca.loc[xid]; recs.append((hp, (x, y, z), (xr.x, xr.y, xr.z)))
        out[mc] = recs
    return out

def compare_to_xtal(s):
    mch = prot_chains(s["atoms"])[:2]; xch = list(XTAL_CA)[:2]
    best = None
    for xm in (xch, xch[::-1]):
        cm = dict(zip(mch, xm)); mp = matched_pairs(s, cm)
        P = np.array([r[1] for c in mch for r in mp[c]]); Q = np.array([r[2] for c in mch for r in mp[c]])
        R, t = kabsch(P, Q); d = rmsd(apply(R, t, P), Q)
        if best is None or d < best[0]: best = (d, cm, mp, R, t)
    d_dimer, cm, mp, R, t = best
    # fit on chain A only, then measure chain B (tests relative placement of the two copies)
    A = np.array([r[1] for r in mp[mch[0]]]); QA = np.array([r[2] for r in mp[mch[0]]])
    RA, tA = kabsch(A, QA); dA = rmsd(apply(RA, tA, A), QA)
    B = np.array([r[1] for r in mp[mch[1]]]); QB = np.array([r[2] for r in mp[mch[1]]])
    dB_afterA = rmsd(apply(RA, tA, B), QB)
    dev = []
    for c in mch:
        P = np.array([r[1] for r in mp[c]]); Q = np.array([r[2] for r in mp[c]])
        for r, dd in zip(mp[c], np.linalg.norm(apply(R, t, P) - Q, axis=1)): dev.append((c, r[0], dd))
    return dict(rmsd_dimer=d_dimer, rmsd_chainA_alone=dA, rmsd_chainB_after_Afit=dB_afterA, n_ca=len(P) * 2, chain_map=cm), pd.DataFrame(dev, columns=["chain", "human_pos", "dev"])

def zinc_distances(atoms, ligands, resid_of):
    zn = atoms[atoms.comp == "ZN"]; res = []
    prot = atoms[atoms.group == "ATOM"]
    side = prot[~prot.atom.isin(["N", "CA", "C", "O", "OXT"]) & prot.atom.str[0].isin(["N", "O"])]
    for _, z in zn.iterrows():
        pz = np.array([z.x, z.y, z.z]); cand = {}
        for c in prot_chains(atoms):
            ds = []
            for L in ligands:
                a = side[(side.chain == c) & (side.resid == resid_of(L))]
                ds.append(np.linalg.norm(a[["x", "y", "z"]].values - pz, axis=1).min() if len(a) else np.nan)
            cand[c] = ds
        c = min(cand, key=lambda k: np.nanmean(cand[k]) if not np.all(np.isnan(cand[k])) else 1e9)
        res.append(dict(zn_chain=z.chain, ligand_chain=c, **{f"d_{L}": round(d, 2) for L, d in zip(ligands, cand[c])}))
    return res

def interface_residues(atoms, cutoff=CONTACT_CUTOFF):
    prot = atoms[atoms.group == "ATOM"]; ch = prot_chains(atoms)[:2]
    A = prot[prot.chain == ch[0]]; B = prot[prot.chain == ch[1]]
    XA, XB = A[["x", "y", "z"]].values, B[["x", "y", "z"]].values
    dmin_a = np.full(len(XA), 1e9); dmin_b = np.full(len(XB), 1e9)
    for i in range(0, len(XA), 400):
        D = np.linalg.norm(XA[i:i + 400, None, :] - XB[None, :, :], axis=2)
        dmin_a[i:i + 400] = D.min(1); dmin_b = np.minimum(dmin_b, D.min(0))
    return set(A.resid[dmin_a <= cutoff].astype(int)), set(B.resid[dmin_b <= cutoff].astype(int))

xt_if_a, xt_if_b = interface_residues(XTAL_ATOMS)
XT_IF = {r + XTAL_OFFSET for r in xt_if_a | xt_if_b}     # human UniProt numbering
XT_ZN = zinc_distances(XTAL_ATOMS, HUMAN_LIG, lambda L: L - XTAL_OFFSET)
print("3LGD interface residues (either chain, <=%.1f A):" % CONTACT_CUTOFF, len(XT_IF))
print("3LGD Zn-ligand distances:", XT_ZN)

In [ ]:
res, DEVS, rows = {}, {}, []
for j, s in STRUCT.items():
    off = UP_OFFSET[(s["species"], s["construct"])]
    r, dev = compare_to_xtal(s); DEVS[j] = dev
    ifa, ifb = interface_residues(s["atoms"])
    if_h = {human_pos_of(s["species"], int(x) + off) for x in ifa | ifb} - {None}
    # restrict the crystal interface to residues that exist in both
    cov = set(dev.human_pos.astype(int)); xt = XT_IF & cov; md_ = if_h & cov
    jac = len(xt & md_) / max(1, len(xt | md_))
    zd = zinc_distances(s["atoms"], ZN_LIGANDS[s["species"]], lambda L, off=off: L - off) if s["zinc"] == "Zn" else []
    zmax = max([v for z in zd for k, v in z.items() if k.startswith("d_")] or [np.nan])
    rows.append(dict(job=j, species=s["species"], zinc=s["zinc"], template=s["template"], seed=s["seed"], **{k: round(v, 2) if isinstance(v, float) else v for k, v in r.items() if k != "chain_map"},
                     n_if_model=len(md_), n_if_3LGD=len(xt), interface_jaccard=round(jac, 2), zn_ligand_max_dist=round(zmax, 2) if zmax == zmax else None))
    res[j] = dict(zinc=zd)
XCOMP = pd.DataFrame(rows)
print("Comparison with 3LGD (human numbering; pig compared on the residues aligned to human):")
display(XCOMP)
for j, v in res.items():
    if v["zinc"]: print(j, "Zn to ligand side-chain N/O distances (A):", v["zinc"])

**How to read this table**
- `rmsd_dimer`: average CA distance after best overlay of the whole dimer. Roughly 1 to 2 A means the prediction reproduces the crystal dimer. A large value alone does not mean a wrong fold; check the next two.
- `rmsd_chainA_alone`: how well one chain matches the crystal.
- `rmsd_chainB_after_Afit`: if this is much larger than `rmsd_chainA_alone`, the two copies are arranged differently from the crystal.
- `interface_jaccard`: overlap of interface residues with 3LGD (1.0 = identical).
- Zn ligand distances should be about 2.0 to 2.3 A in a correct zinc site (3LGD values printed above for reference).
- Pig is compared with the human crystal only on residues aligned to human. It is not an experimental pig structure.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))
for j, d in DEVS.items():
    g = d.groupby("human_pos").dev.mean()
    ax.plot(g.index, g.values, lw=.8, label=j, ls="-" if STRUCT[j]["species"] == "human" else "--")
for lo, hi, c in [(30, 100, "#ccd"), (127, 185, "#dcc"), (204, 211, "#cdc")]: ax.axvspan(lo, hi, color=c, alpha=.5)
for z in HUMAN_LIG: ax.axvline(z, c="r", lw=.5)
ax.set_xlabel("human UniProt position"); ax.set_ylabel("CA distance to 3LGD (A)"); ax.legend(fontsize=6, ncol=2)
ax.set_title("Per-residue deviation from 3LGD after overlaying the dimer"); plt.tight_layout(); plt.show()

## 6b. Does using PDB templates change the answer?

In [ ]:
cols = ["job", "template", "iptm", "ptm", "mean_pae_between"]
print("Scores:"); display(BEST[cols].round(3))
print("3LGD comparison, template vs noTemplate:")
display(XCOMP.groupby(["species", "template"])[["rmsd_dimer", "rmsd_chainA_alone", "rmsd_chainB_after_Afit", "interface_jaccard"]].mean().round(2))

AlphaFold Server uses PDB templates by default. For pig the top template was 3LGD (the human crystal structure), so agreement with 3LGD is not independent evidence. If the noTemplate pig model still matches human closely, the similarity comes from the sequence, not the template.

## 7. Seed variability (same species / zinc / construct, top model per seed)

In [ ]:
def dimer_ca(s):
    ch = prot_chains(s["atoms"])[:2]; return np.vstack([s["ca"][c][["x", "y", "z"]].values for c in ch])

rows = []
groups = {}
for j, s in STRUCT.items(): groups.setdefault((s["species"], s["construct"], s["zinc"], s["template"]), []).append(j)
for key, js in groups.items():
    for a, b in itertools.combinations(js, 2):
        A, B = dimer_ca(STRUCT[a]), dimer_ca(STRUCT[b])
        if len(A) != len(B): continue
        h = len(A) // 2; Bsw = np.vstack([B[h:], B[:h]])      # also try swapped chain order
        best = min(rmsd(apply(*kabsch(A, X), A), X) for X in (B, Bsw))
        rows.append(dict(species=key[0], construct=key[1], zinc=key[2], template=key[3], job_a=a, job_b=b, rmsd_A=round(best, 2)))
SEEDVAR = pd.DataFrame(rows)
print("Pairwise RMSD between seeds (small = reproducible):"); display(SEEDVAR if len(SEEDVAR) else "need at least 2 seeds of the same setup")

## 8. Human vs pig (predicted vs predicted, aligned positions only)

In [ ]:
def best_of(species, zinc="Zn", construct="mature", template="template"):
    c = BEST[(BEST.species == species) & (BEST.zinc == zinc) & (BEST.construct == construct) & (BEST.template == template)]
    return None if c.empty else c.sort_values("ranking_score").iloc[-1].job

def by_h(s, ch):
    off = UP_OFFSET[(s["species"], s["construct"])]
    return {human_pos_of(s["species"], int(r) + off): (x, y, z) for r, x, y, z in zip(s["ca"][ch].resid, s["ca"][ch].x, s["ca"][ch].y, s["ca"][ch].z)}

def human_vs_pig(hj, pj):
    H, P = STRUCT[hj], STRUCT[pj]; hc, pc = prot_chains(H["atoms"])[:2], prot_chains(P["atoms"])[:2]
    best = None
    for pm_ in (pc, pc[::-1]):
        Ps, Qs, keys = [], [], []
        for hch, pch in zip(hc, pm_):
            a, b = by_h(H, hch), by_h(P, pch)
            for k in sorted(set(a) & set(b) - {None}): Ps.append(b[k]); Qs.append(a[k]); keys.append((hch, k))
        Ps, Qs = np.array(Ps), np.array(Qs); R, t = kabsch(Ps, Qs); d = np.linalg.norm(apply(R, t, Ps) - Qs, axis=1)
        if best is None or d.mean() < best[0].mean(): best = (d, keys)
    d, keys = best
    HP = pd.DataFrame(dict(chain=[k[0] for k in keys], human_pos=[k[1] for k in keys], dev=d))
    return HP, float(np.sqrt((d ** 2).mean()))

HVP = {}
for tmpl in ("template", "noTemplate"):
    hj, pj = best_of("human", template=tmpl), best_of("pig", template=tmpl)
    if hj and pj and hj in STRUCT and pj in STRUCT:
        HVP[tmpl] = human_vs_pig(hj, pj); print(f"[{tmpl}] {hj}  vs  {pj}: RMSD = {HVP[tmpl][1]:.2f} A over {len(HVP[tmpl][0]) // 2} aligned positions x 2 chains")
    else:
        print(f"[{tmpl}] need a finished human Zn and pig Zn job. Found:", hj, pj)
if HVP:
    fig, ax = plt.subplots(figsize=(14, 3.8))
    for tmpl, (HP, _) in HVP.items():
        g = HP.groupby("human_pos").dev.mean(); ax.plot(g.index, g.values, lw=.8, label=f"human vs pig, {tmpl}")
    HP0 = HVP.get("noTemplate", HVP.get("template"))[0]; g0 = HP0.groupby("human_pos").dev.mean()
    diff = [p for p in g0.index if IDENT.get(p) is False]; ax.scatter(diff, g0.loc[diff], s=6, c="tab:red", label="residue differs in pig", zorder=3)
    for lo, hi, c in [(30, 100, "#ccd"), (127, 185, "#dcc"), (204, 211, "#cdc")]: ax.axvspan(lo, hi, color=c, alpha=.5)
    ax.set_xlabel("human UniProt position"); ax.set_ylabel("CA distance human vs pig model (A)"); ax.legend(fontsize=7); plt.tight_layout(); plt.show()
    for tmpl, (HP, _) in HVP.items():
        print(f"Top deviating positions ({tmpl}):"); display(HP.groupby("human_pos").dev.mean().sort_values(ascending=False).head(12).round(2).to_frame("dev_A").T)

**Caution:** a human-pig difference in two predictions is only a clue. Check that it is larger than the seed-to-seed spread in section 7, and that confidence (pLDDT / PAE) is high in that region, before treating it as real.

## 8b. Confidence (pLDDT / PAE) in chosen regions, human numbering

In [ ]:
REGIONS = {"N-term 30-33": range(30, 34), "151-160 (insertion zone)": range(151, 161), "281": [281], "347": [347], "C-term 511": [511], "Whole chain (reference)": range(30, 511)}
HUM2PIG_ = {v: k for k, v in PIG2HUM.items()}
REGION_ROWS = []
for name, models in JOBS.items():
    species, construct, zinc, seed, template = parse_meta(name); off = UP_OFFSET[(species, construct)]
    for idx, fp in sorted(models.items()):
        if not (fp["full"].exists() and fp["cif"].exists()): continue
        fd = load_json(fp["full"]); pae = np.array(fd["pae"]); ch = np.array(fd["token_chain_ids"]); rid = np.array(fd["token_res_ids"])
        ca_ = ca_table(read_atoms(fp["cif"]))["A"]; pl = dict(zip(ca_.resid.astype(int), ca_.b))
        a_, b_ = ch == "A", ch == "B"
        for rn, rr in REGIONS.items():
            ids = [(h if species == "human" else HUM2PIG_.get(h)) for h in rr]; ids = [i - off for i in ids if i is not None]
            if not ids: continue
            sel = a_ & np.isin(rid, ids)
            REGION_ROWS.append(dict(job=name, species=species, template=template, model=idx, region=rn, n_residues=len(ids),
                                    plddt=np.mean([pl[i] for i in ids if i in pl]), pae_within_chain=pae[np.ix_(sel, a_)].mean(), pae_to_other_chain=pae[np.ix_(sel, b_)].mean()))
REGCONF = pd.DataFrame(REGION_ROWS)
top = REGCONF.merge(BEST[["job", "model"]], on=["job", "model"])
display(top.groupby(["region", "species"])[["plddt", "pae_within_chain", "pae_to_other_chain"]].agg(["mean", "min", "max"]).round(1))

Low pLDDT and high PAE mean AlphaFold is unsure of that region (typically flexible loops and chain ends). High confidence with a human-pig difference is a stronger candidate for a real difference.

## 9. Save results

In [ ]:
MODELS.to_csv(OUT_DIR / "af3_all_model_scores.csv", index=False)
BEST.to_csv(OUT_DIR / "af3_top_model_per_job.csv", index=False)
XCOMP.to_csv(OUT_DIR / "af3_vs_3LGD_comparison.csv", index=False)
if len(SEEDVAR): SEEDVAR.to_csv(OUT_DIR / "af3_seed_variability.csv", index=False)
if HVP: pd.concat([HP.assign(template=t) for t, (HP, _) in HVP.items()]).to_csv(OUT_DIR / "af3_human_vs_pig_per_residue.csv", index=False)
REGCONF.to_csv(OUT_DIR / "region_confidence_all_models.csv", index=False)
print("saved to", OUT_DIR)